# HELM — Training Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimchev/HELM/blob/master/notebooks/demo_training.ipynb)

End-to-end training of the HELM model on a remote-sensing multi-label classification dataset.
This notebook covers data preparation, model construction, training, and evaluation.

## 1. Setup

In [ ]:
%cd /content
!rm -rf HELM
!git clone https://github.com/marjanstoimchev/HELM.git
%cd HELM
!pip install -q -r requirements.txt

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html 2>/dev/null || echo "PyG extensions installed via fallback"
print("Setup complete!")

In [ ]:
import os, sys, gc, glob, yaml
import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning import seed_everything
import matplotlib.pyplot as plt

sys.path.insert(0, ".")

from data.dataset_pipeline import DatasetPipeline
from data.hierarchy import create_edge_index
from datamodules.base_datamodule import BaseDataModule
from models.model import h_deit_base_embedding
from augmentations import Preprocess
from callbacks import ModelCheckpoint_, EarlyStopping_, RichProgressBar_
from utils.utils import Dotdict, calculate_metrics, predict
from trainers import get_trainer_class

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Configuration

In [ ]:
DATASET = "ucm"
DATASET_CONFIG = f"configs/dataset/{DATASET}.yaml"
METHOD = "hmlc-sl-graph-byol"     # HELM supervised (graph + BYOL)
FRACTION_LABELED = 1.0
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-4
MAX_LR = 3e-4
PATIENCE = 5
PRETRAINED = True
DEVICES = [0]

SKIP_TRAINING = True   # True = download pre-trained + evaluate, False = train from scratch

# Pre-trained model on Google Drive (UCM, hmlc-sl-graph-byol, 100% labeled)
GDRIVE_FILE_ID = "106kboTGXisVJ6RWHmpGeBfkWyTE3vHqr"

with open(f"configs/method/{METHOD}.yaml") as f:
    method_cfg = yaml.safe_load(f)

LEARNING_TASK = method_cfg["learning_task"]
TRAINING_MODE = method_cfg["training_mode"]
APPLY_EDGES = method_cfg["apply_edges"]
APPLY_BYOL = method_cfg["apply_byol"]

print(f"Dataset:       {DATASET}")
print(f"Method:        {METHOD}")
print(f"Training mode: {TRAINING_MODE}")
print(f"Graph (SAGE):  {APPLY_EDGES}")
print(f"BYOL:          {APPLY_BYOL}")
print(f"Fraction:      {int(FRACTION_LABELED * 100)}%")
print(f"Mode:          {'skip training → download + evaluate' if SKIP_TRAINING else 'train from scratch'}")

## 3. Prepare data

In [ ]:
seed_everything(SEED, workers=True)

pipeline = DatasetPipeline(
    yaml_path=DATASET_CONFIG, seed=SEED,
    cache_dir="./Datasets/mlc_datasets_npy",
)

fraction = FRACTION_LABELED if TRAINING_MODE == "ssl" else None
outputs = pipeline.run_pipeline(fraction_labeled=fraction)
if TRAINING_MODE == "sl":
    outputs.pop("U", None)

num_leaves = pipeline.num_classes
num_classes = num_leaves if LEARNING_TASK == "mlc" else pipeline.num_classes_extended

edge_index = None
if APPLY_EDGES and LEARNING_TASK == "hmlc":
    edge_index = create_edge_index(hierarchy=pipeline.label_to_predecessors)

transforms = Preprocess()
datamodule = BaseDataModule(outputs, batch_size=BATCH_SIZE, num_workers=2, transforms=transforms)

config = Dotdict({
    "training": {
        "lr": LR, "head_lr": LR, "max_lr": MAX_LR,
        "apply_scheduler": True, "epochs": EPOCHS, "min_epochs": 1,
        "patience": PATIENCE, "lr_schedule_patience": 5,
        "accumulate_grad_batches": 5, "deterministic": True, "log_every_n_steps": 1,
    },
    "dataset": {"name": DATASET, "folder_name": DATASET, "num_classes": num_leaves},
})

print(f"Classes: {num_classes} total ({num_leaves} leaves)")
print(f"Train: {len(outputs['X'][0])}, Val: {len(outputs['X_val'])}, Test: {len(outputs['X_te'])}")
if "U" in outputs:
    print(f"Unlabeled: {len(outputs['U'])}")

## 4. Train or load model

- `SKIP_TRAINING = True`: downloads pre-trained weights from Google Drive and runs inference
- `SKIP_TRAINING = False`: trains from scratch, then evaluates

In [ ]:
frac_int = int(100 * FRACTION_LABELED)
save_dir = f"saved_models/{DATASET}/{METHOD}/fraction_{frac_int}/seed_{SEED}"
model_path = os.path.join(save_dir, "model.ckpt")

backbone = h_deit_base_embedding(num_classes=num_classes, pretrained=PRETRAINED if not SKIP_TRAINING else False)
trainer_cls = get_trainer_class(TRAINING_MODE, LEARNING_TASK, APPLY_EDGES, APPLY_BYOL)
lightning_model = trainer_cls(config, backbone, num_leaves, LEARNING_TASK, edge_index)

if not SKIP_TRAINING:
    # ── Train from scratch ───────────────────────────────────────────────
    print(f"Model: {trainer_cls.__name__}")
    print(f"Parameters: {sum(p.numel() for p in lightning_model.parameters()) / 1e6:.1f}M\n")

    trainer = L.Trainer(
        accelerator="auto", devices=DEVICES, strategy="auto",
        precision="16-mixed", max_epochs=EPOCHS, min_epochs=1,
        accumulate_grad_batches=5, enable_checkpointing=True,
        enable_model_summary=True, num_sanity_val_steps=0, log_every_n_steps=1,
        callbacks=[
            RichProgressBar_(),
            EarlyStopping_(metric="val_loss", mode="min", patience=PATIENCE),
            ModelCheckpoint_(dirpath=save_dir, metric="val_loss", mode="min"),
        ],
    )
    trainer.fit(lightning_model, datamodule=datamodule)

    os.makedirs(save_dir, exist_ok=True)
    torch.save(lightning_model.state_dict(), os.path.join(save_dir, "final_model.pt"))
    print(f"\nModel saved to {save_dir}/final_model.pt")
else:
    # ── Download pre-trained checkpoint ───────────────────────────────────
    if not os.path.exists(model_path):
        !pip install -q gdown
        import gdown
        os.makedirs(save_dir, exist_ok=True)
        print("Downloading pre-trained HELM checkpoint...")
        gdown.download(id=GDRIVE_FILE_ID, output=model_path, quiet=False)

    # Load checkpoint — map to current device (handles GPU→CPU and multi-GPU→single-GPU)
    map_loc = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    ckpt = torch.load(model_path, map_location=map_loc, weights_only=False)
    lightning_model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded pre-trained checkpoint from {model_path}")

    accel = "gpu" if torch.cuda.is_available() else "cpu"
    trainer = L.Trainer(
        accelerator=accel,
        devices=1 if accel == "cpu" else [0],
        strategy="auto",
        enable_model_summary=False,
    )

## 5. Evaluate

In [ ]:
Y = predict(trainer, lightning_model, datamodule)
df_metrics = calculate_metrics(Y)
df_metrics.columns = ["Value"]

print(f"\n{'='*45}")
print(f"  {METHOD} on {DATASET} ({frac_int}% labeled)")
print(f"{'='*45}")
display(df_metrics.style.format({"Value": "{:.4f}"}))